# 08 — Multi-state survival panel

**Input:** `Results/07.xlsx`
**Output:** `Results/08.csv`, `Results/df_mstate_ready.csv`

Reshapes the wide panel into transition spells (Tstart, Tstop, from, to) in
counting-process format. The exported model dataset keeps sequential
advancements, censored spells and cancellations, and drops the non-linear
trajectories (forward jumps, backward steps) that would violate the assumption
of sequential progression; those are reported separately in the Supplementary
Information.

Zero-length spells, an artefact of the biennial reporting cadence, receive a
deterministic offset so that continuous-time competing-risks models converge.

In [1]:
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter

In [2]:
df = pd.read_excel("Results/07.xlsx", header=[0,1], index_col=0)

In [3]:
df_meta = df['meta'].copy()

overlapping_cols = [col for col in df_meta.columns if col in df.columns.get_level_values(1)]

df_temporal = df.drop(columns='meta', level=0) \
                .drop(columns=['Inv_index', 'Project_ID'] + overlapping_cols, level=1, errors='ignore')

df_long = df_temporal.stack(level=0, future_stack=True).rename_axis(index=['Original_Index', 'Year']).reset_index()

df_final_long = pd.merge(df_long, df_meta, left_on='Original_Index', right_index=True)

In [4]:
df_final_long['State'] = df_final_long.groupby('Inv_index')['Inv_Status'].ffill()
df_clean = df_final_long.sort_values(by=['Inv_index', 'Year']).dropna(subset=['State']).copy()

df_clean['Prev_State'] = df_clean.groupby('Inv_index')['State'].shift(1)

mask_transition = (df_clean['State'] != df_clean['Prev_State']) & df_clean['Prev_State'].notna()
mask_first = df_clean['Prev_State'].isna()
mask_last = ~df_clean.duplicated(subset=['Inv_index'], keep='last')

df_events = df_clean[mask_transition | mask_first | mask_last].copy()

df_events['Tstop'] = df_events['Year'].astype(float)
df_events['Tstart'] = df_events.groupby('Inv_index')['Tstop'].shift(1)
df_events['from'] = df_events.groupby('Inv_index')['State'].shift(1)
df_events['to'] = df_events['State']

df_final = df_events.dropna(subset=['from', 'to']).copy()
df_final['from'] = df_final['from'].astype(int)
df_final['to'] = df_final['to'].astype(int)

df_final = df_final[~df_final['from'].isin([5, 6])].copy()

df_final = df_final.sort_values(by=['Inv_index', 'Tstart', 'Tstop'])
time_jitter = df_final.groupby(['Inv_index', 'Tstart']).cumcount() * 0.1
df_final['Tstart'] = df_final['Tstart'] + time_jitter

mask_instant = (df_final['Tstop'] <= df_final['Tstart'])
df_final.loc[mask_instant, 'Tstop'] = df_final.loc[mask_instant, 'Tstart'] + 0.1
df_final = df_final[df_final['Tstop'] > df_final['Tstart']].copy()


In [5]:
# Create the NewDirective flag (Post-2016, effective from 2018)
# If the phase starts from 2018 onwards
df_final['NewDirective'] = np.where(df_final['Tstart'] >= 2018, 1, 0)

# Create the Covid19 flag (Window: 2022 - 2024)
# A phase is affected if it overlaps with the 2022-2024 period
df_final['Covid19'] = np.where(
    (df_final['Tstart'] <= 2024) & (df_final['Tstop'] >= 2022), 
    1, 0
)

df_final = df_final.rename(columns={
    'Inv_Technology[AC/DC]': 'Inv_Technology',
    'Inv_Line length [km]': 'Inv_Line_Length_km',
    'Inv_Capacity [MW]': 'Capacity_MW', 
    'Inv_Voltage [kV]': 'Voltage_kV',
})

In [6]:
# --- EXPORT: Raw data for Raincloud and Markov Models ---
covariates = [
    'Inv_Line_Length_km', 
    'Inv_Technology', 
    'Capacity_MW',
    'Voltage_kV',
    'Inv_Element type', 
    'Project_Country_ISO3',
    'Project_N_Countries',
    'Project_Region',
    'Project_Jurisdiction',
    'Infr_Type',
    'Inv_Element_Category',
    'Inv_Environment', 
    'Covid19',
    'NewDirective'
]
existing_covariates = [c for c in covariates if c in df_final.columns]

In [7]:
df_export_08 = df_final[['Inv_index', 'Tstart', 'Tstop', 'from', 'to'] + existing_covariates].copy()
df_export_08.to_csv("Results/08.csv", index=False)


In [8]:
df_export_08.to_excel('Results/08.xlsx')

### Cox Model data

In [9]:
df_final['Duration'] = df_final['Tstop'] - df_final['Tstart']

df_mstate = df_final[~df_final['from'].isin([5, 6])].copy()

mask_valid_segment = ((df_mstate['to'] - df_mstate['from'] == 1) | 
                      (df_mstate['to'] == df_mstate['from']) | 
                      (df_mstate['to'] == 6))

df_mstate = df_mstate[mask_valid_segment].copy()

def define_status_segment(row):
    """
    1 = Success (Sequential step)
    2 = Failure (Cancellation)
    0 = Censored (Stuck)
    """
    if row['to'] == 6:
        return 2  
    elif row['to'] - row['from'] == 1:
        return 1  
    else:
        return 0  

df_mstate['status'] = df_mstate.apply(define_status_segment, axis=1)



In [10]:
columns_to_export = ['Inv_index', 'Duration', 'from', 'to', 'status'] + existing_covariates
df_mstate_export = df_mstate[columns_to_export].copy()


In [11]:
df_mstate_export.columns = df_mstate_export.columns.str.replace(r'\[|\]|\s|\/', '.', regex=True)
df_mstate_export.to_csv("Results/df_mstate_ready.csv", index=False)
print(f"Saved Results/df_mstate_ready.csv. Final rows: {len(df_mstate_export)}")

Saved Results/df_mstate_ready.csv. Final rows: 1201
